In [9]:
import requests
import pandas as pd
import time
from io import StringIO
from datetime import datetime

PARK_ID = 160
START_YEAR = 2023
END_YEAR = 2025  # tot nu
BASE_OUTPUT = "efteling_rides"

def to_number(series):
    return (series.astype(str)
                  .str.replace(",", ".", regex=False)
                  .str.extract(r"(\d+\.?\d*)")[0]
                  .astype(float))

today = datetime.today()

for year in range(START_YEAR, END_YEAR + 1):
    start = f"{year}-01-01"
    if year == today.year:
        end = today.strftime("%Y-%m-%d")
    else:
        end = f"{year}-12-31"

    rows = []

    for date in pd.date_range(start, end):
        url = f"https://queue-times.com/parks/{PARK_ID}/calendar/{date:%Y/%m/%d}"
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
        html = response.text

        tables = pd.read_html(StringIO(html))

        avg = mx = up = None

        for table in tables:
            cols = [str(c).lower() for c in table.columns]
            if not cols or "ride" not in cols[0]:
                continue

            if len(cols) > 1:
                if "average" in cols[1]:
                    avg = table.rename(columns={table.columns[0]: "ride",
                                                table.columns[1]: "avg_queue_min"})
                    avg["avg_queue_min"] = to_number(avg["avg_queue_min"])
                elif "maximum" in cols[1] or "max" in cols[1]:
                    mx = table.rename(columns={table.columns[0]: "ride",
                                               table.columns[1]: "max_queue_min"})
                    mx["max_queue_min"] = to_number(mx["max_queue_min"])
                elif "uptime" in cols[1]:
                    up = table.rename(columns={table.columns[0]: "ride",
                                               table.columns[1]: "uptime_pct"})
                    up["uptime_pct"] = to_number(up["uptime_pct"])

        df = None
        for part in (avg, mx, up):
            if part is not None:
                df = part if df is None else df.merge(part, on="ride", how="outer")

        if df is not None and not df.empty:
            df.insert(0, "date", date.date().isoformat())
            df.insert(1, "park_id", PARK_ID)
            rows.append(df)
        
        print(f"Scraped data for {date.date().isoformat()}")
        time.sleep(1)  # 1 seconde wachten om niet te snel te scrapen

    year_df = pd.concat(rows, ignore_index=True)
    output_file = f"{BASE_OUTPUT}_{year}.csv"
    year_df.to_csv(output_file, index=False)
    print(f"✅ {year} data saved to {output_file}")


Scraped data for 2023-01-01
Scraped data for 2023-01-02
Scraped data for 2023-01-03
Scraped data for 2023-01-04
Scraped data for 2023-01-05
Scraped data for 2023-01-06
Scraped data for 2023-01-07
Scraped data for 2023-01-08
Scraped data for 2023-01-09
Scraped data for 2023-01-10
Scraped data for 2023-01-11
Scraped data for 2023-01-12
Scraped data for 2023-01-13
Scraped data for 2023-01-14
Scraped data for 2023-01-15
Scraped data for 2023-01-16
Scraped data for 2023-01-17
Scraped data for 2023-01-18
Scraped data for 2023-01-19
Scraped data for 2023-01-20
Scraped data for 2023-01-21
Scraped data for 2023-01-22
Scraped data for 2023-01-23
Scraped data for 2023-01-24
Scraped data for 2023-01-25
Scraped data for 2023-01-26
Scraped data for 2023-01-27
Scraped data for 2023-01-28
Scraped data for 2023-01-29
Scraped data for 2023-01-30
Scraped data for 2023-01-31
Scraped data for 2023-02-01
Scraped data for 2023-02-02
Scraped data for 2023-02-03
Scraped data for 2023-02-04
Scraped data for 202

OSError: [Errno 30] Read-only file system: 'efteling_rides_2023.csv'